# 04 – Explore municipality boundaries (from DeSO)

**Goal:** get a polygon for each of Sweden's 290 municipalities, so every grid cell and restaurant can be placed in a municipality.

**Why DeSO?** SCB's map service has no municipality layer. But every DeSO area (*demografiskt statistikområde*) lies inside exactly **one** municipality, and the **first four characters of the DeSO code are the municipality code**. Merging the DeSO areas per code gives the municipalities – from the same source, year and coordinate system as the population grid.

**Data:** `raw.scb_deso`, loaded by `hamburgarkartan.boundaries` (layer `DeSO_2025`, SWEREF 99 TM).

Run `uv run python -m hamburgarkartan.boundaries` first.

In [ ]:
import duckdb
from hamburgarkartan.boundaries import DB_PATH

# read_only = we only look, and other programs can still use the database
con = duckdb.connect(str(DB_PATH), read_only=True)
con.sql("INSTALL spatial; LOAD spatial;")  # needed for geometry functions

## 1. Which columns did SCB give us?

In [ ]:
con.sql("DESCRIBE raw.scb_deso")

**Finding:** fill in – which column holds the DeSO code (e.g. `0180C1010`)? Is there also a municipality code or name?

## 2. A few example rows

In [ ]:
con.sql("SELECT * EXCLUDE (geom) FROM raw.scb_deso LIMIT 5")

## 3. Pick the columns to use
The cell guesses the geometry column and the DeSO code column. **Check the guess** and change it if it's wrong.

In [ ]:
cols = con.sql("DESCRIBE raw.scb_deso").fetchall()  # (name, type, ...)

GEOM = next(name for name, typ, *_ in cols if typ == "GEOMETRY")
CODE = next((name for name, typ, *_ in cols if "deso" in name.lower() and typ == "VARCHAR"), None)

print("geometry column:", GEOM)
print("DeSO code column:", CODE)
# If a guess is wrong, set it by hand, e.g.:  CODE = "desokod"

## 4. How many DeSO areas, and how many municipalities?
The first 4 characters of the DeSO code = municipality code. There should be exactly **290** different ones.

In [ ]:
con.sql(f"""
    SELECT
        count(*)                                  AS deso_areas,
        count(DISTINCT {CODE})                    AS unique_deso_codes,   -- should equal deso_areas
        count(DISTINCT left({CODE}, 4))           AS municipalities,      -- should be 290
        count(*) FILTER (WHERE {CODE} IS NULL)    AS missing_codes
    FROM raw.scb_deso
""")

**Finding:** fill in – number of DeSO areas (about 6 000 expected) and municipalities (290).

## 5. Do the codes match the municipality statistics (TAB628)?
Every municipality code from DeSO should exist in `raw.scb_kommun_stats`, and the other way round. If not, the join in dbt will lose municipalities.

In [ ]:
con.sql(f"""
    WITH deso AS (SELECT DISTINCT left({CODE}, 4) AS code FROM raw.scb_deso),
         stats AS (SELECT DISTINCT Region AS code, Region_label AS name
                   FROM raw.scb_kommun_stats WHERE length(Region) = 4)
    SELECT
        (SELECT count(*) FROM deso)                                           AS codes_in_deso,
        (SELECT count(*) FROM stats)                                          AS codes_in_stats,
        (SELECT count(*) FROM deso WHERE code NOT IN (SELECT code FROM stats)) AS only_in_deso,
        (SELECT count(*) FROM stats WHERE code NOT IN (SELECT code FROM deso)) AS only_in_stats
""")

**Finding:** fill in – `only_in_deso` and `only_in_stats` should both be **0**.

## 6. Is the geometry valid?
Invalid polygons (e.g. self-crossing borders) can break the merge and point-in-polygon tests.

In [ ]:
con.sql(f"""
    SELECT
        ST_GeometryType({GEOM})                      AS type,
        count(*)                                     AS areas,
        count(*) FILTER (WHERE NOT ST_IsValid({GEOM})) AS invalid,
        min(ST_XMin({GEOM})) AS min_x, max(ST_XMax({GEOM})) AS max_x,
        min(ST_YMin({GEOM})) AS min_y, max(ST_YMax({GEOM})) AS max_y
    FROM raw.scb_deso
    GROUP BY 1
""")

**Finding:** fill in – type (polygon / multipolygon), any invalid ones? Coordinates should be SWEREF 99 TM (x ≈ 200 000–950 000, y ≈ 6 100 000–7 700 000).

## 7. Merge DeSO → municipalities (preview)
This is what dbt will do later. `ST_MakeValid` repairs small errors first.

In [ ]:
import time

start = time.perf_counter()
con.sql(f"""
    CREATE OR REPLACE TEMP TABLE kommun_geom AS
    SELECT
        left({CODE}, 4)                         AS kommun_code,
        count(*)                                AS deso_areas,
        ST_Union_Agg(ST_MakeValid({GEOM}))      AS geom
    FROM raw.scb_deso
    GROUP BY 1
""")
print(f"merged in {time.perf_counter() - start:.0f} s")
con.sql("SELECT kommun_code, deso_areas, ST_GeometryType(geom) AS type FROM kommun_geom LIMIT 5")

**Finding:** fill in – how long did the merge take?

## 8. Does the area make sense?
Compare the merged area with SCB's **land** area. DeSO may include lakes and coast, so our area can be **larger** – but never much smaller.

In [ ]:
con.sql("""
    WITH stats AS (
        SELECT Region AS kommun_code, Region_label AS name, value AS land_area_km2
        FROM raw.scb_kommun_stats
        WHERE length(Region) = 4 AND ContentsCode = 'BE0101U3'
    )
    SELECT
        s.name,
        round(ST_Area(k.geom) / 1e6, 1)                   AS polygon_area_km2,
        s.land_area_km2,
        round(ST_Area(k.geom) / 1e6 / s.land_area_km2, 2) AS ratio      -- ≥ 1 expected
    FROM kommun_geom k
    JOIN stats s USING (kommun_code)
    ORDER BY ratio
    LIMIT 10
""")

**Finding:** fill in – are any ratios clearly below 1 (polygon smaller than land area → something is missing)?

## 9. Quick map
Each municipality gets a colour. It should look like Sweden, with no holes.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon
import json

# simplify (100 m tolerance) so the plot is fast; only for drawing
shapes = con.sql("""
    SELECT kommun_code, ST_AsGeoJSON(ST_Simplify(geom, 100)) AS gj FROM kommun_geom
""").fetchall()

patches = []
for _, gj in shapes:
    g = json.loads(gj)
    polys = g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]
    for poly in polys:
        patches.append(MplPolygon(poly[0], closed=True))   # outer ring only

fig, ax = plt.subplots(figsize=(5, 9))
ax.add_collection(PatchCollection(patches, cmap="tab20", array=range(len(patches)),
                                  edgecolor="white", linewidth=0.2))
ax.autoscale()
ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title(f"{len(shapes)} municipalities from DeSO")
plt.show()

**Finding:** fill in – does it look like Sweden with 290 municipalities and no gaps?

## Summary
- DeSO areas: _fill in_ → municipalities after merge: _fill in_ (should be 290)
- Codes match TAB628: _yes / no_
- Invalid geometries: _fill in_ (repaired with `ST_MakeValid`)
- Merge time: _fill in_ s

**Next:** all four sources are loaded → merge `feature/ingestion` into `main`, then start dbt (`feature/dbt-staging`). The merge DeSO → municipality becomes the dbt model `stg_scb__kommuner`.

In [ ]:
con.close()